In [8]:
!pip install yfinance gym numpy pandas torch matplotlib



[notice] A new release of pip available: 22.3 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import yfinance as yf
import numpy as np
import pandas as pd
import gym
from gym import spaces
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque
import matplotlib.pyplot as plt


In [10]:
# Download data (example: Apple)
df = yf.download("AAPL", start="2010-01-01", end="2025-01-01")
prices = df["Close"].values


[*********************100%***********************]  1 of 1 completed


In [11]:
class PriceLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1])

# Prepare LSTM training data
window = 30
X, y = [], []
for i in range(len(prices) - window):
    X.append(prices[i:i+window])
    y.append(prices[i+window])

X = np.array(X).reshape(-1, window, 1)
y = np.array(y)

train_size = int(len(X)*0.8)
X_train, y_train = X[:train_size], y[:train_size]

train_tensor = torch.tensor(X_train, dtype=torch.float32)
target_tensor = torch.tensor(y_train, dtype=torch.float32).reshape(-1,1)

# Train LSTM
lstm_model = PriceLSTM(1, 32)
criterion = nn.MSELoss()
optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)

for epoch in range(50):
    lstm_model.train()
    optimizer.zero_grad()
    out = lstm_model(train_tensor)
    loss = criterion(out, target_tensor)
    loss.backward()
    optimizer.step()

print("LSTM training complete.")


LSTM training complete.


In [5]:
class PriceLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1])

# Prepare LSTM training data
window = 30
X, y = [], []
for i in range(len(prices) - window):
    X.append(prices[i:i+window])
    y.append(prices[i+window])

X = np.array(X).reshape(-1, window, 1)
y = np.array(y)

train_size = int(len(X)*0.8)
X_train, y_train = X[:train_size], y[:train_size]

train_tensor = torch.tensor(X_train, dtype=torch.float32)
target_tensor = torch.tensor(y_train, dtype=torch.float32).reshape(-1,1)

# Train LSTM
lstm_model = PriceLSTM(1, 32)
criterion = nn.MSELoss()
optimizer = optim.Adam(lstm_model.parameters(), lr=0.001)

for epoch in range(50):
    lstm_model.train()
    optimizer.zero_grad()
    out = lstm_model(train_tensor)
    loss = criterion(out, target_tensor)
    loss.backward()
    optimizer.step()

print("LSTM training complete.")


In [12]:
class TradingEnv(gym.Env):
    def __init__(self, price_data, lstm):
        super().__init__()
        self.prices = price_data
        self.lstm = lstm
        self.step_idx = window
        self.action_space = spaces.Discrete(3)  # Hold=0, Buy=1, Sell=2
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(3,), dtype=np.float32)
        self.inventory = 0
        self.cash = 10000

    def reset(self):
        self.step_idx = window
        self.inventory = 0
        self.cash = 10000
        return self._get_state()

    def _get_state(self):
        seq = self.prices[self.step_idx-window:self.step_idx]
        pred_price = self.lstm(torch.tensor(seq.reshape(1, window, 1), dtype=torch.float32)).item()
        price = self.prices[self.step_idx]
        return np.array([price, float(pred_price), self.inventory], dtype=np.float32)

    def step(self, action):
        price = self.prices[self.step_idx]
        # Execute action
        if action == 1:  # Buy
            self.inventory += 1
            self.cash -= price
        elif action == 2 and self.inventory > 0:  # Sell
            self.inventory -= 1
            self.cash += price

        self.step_idx += 1
        done = self.step_idx >= len(self.prices)-1
        reward = self.cash + self.inventory * price
        return self._get_state(), reward, done, {}


In [13]:
class DQNAgent(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim)
        )

    def forward(self, x):
        return self.fc(x)

env = TradingEnv(prices, lstm_model)
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

agent = DQNAgent(state_dim, action_dim)
optimizer = optim.Adam(agent.parameters(), lr=0.001)
criterion = nn.MSELoss()

memory = deque(maxlen=2000)
gamma = 0.99
epsilon = 1.0
epsilon_min = 0.01
epsilon_decay = 0.995
batch_size = 32


In [14]:
for episode in range(200):
    state = env.reset()
    total_reward = 0
    done = False

    while not done:
        if random.random() < epsilon:
            action = random.randrange(action_dim)
        else:
            with torch.no_grad():
                q_vals = agent(torch.tensor(state, dtype=torch.float32))
                action = torch.argmax(q_vals).item()

        next_state, reward, done, _ = env.step(action)
        memory.append((state, action, reward, next_state, done))
        state = next_state
        total_reward += reward

        # Train from memory
        if len(memory) >= batch_size:
            batch = random.sample(memory, batch_size)
            states, actions, rewards, next_states, dones = zip(*batch)

            states = torch.tensor(states, dtype=torch.float32)
            next_states = torch.tensor(next_states, dtype=torch.float32)
            actions = torch.tensor(actions)
            rewards = torch.tensor(rewards)
            dones = torch.tensor(dones, dtype=torch.float32)

            curr_q = agent(states).gather(1, actions.unsqueeze(1)).squeeze()
            next_q = agent(next_states).max(1)[0].detach()
            expected_q = rewards + gamma * next_q * (1 - dones)

            loss = criterion(curr_q, expected_q)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    if episode % 10 == 0:
        print(f"Episode {episode}, Reward: {total_reward:.1f}")


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (3,) + inhomogeneous part.

In [15]:
state = env.reset()

while True:
    with torch.no_grad():
        action = torch.argmax(agent(torch.tensor(state, dtype=torch.float32))).item()
    signal = ["Hold", "Buy", "Sell"][action]
    print(signal)
    state, _, done, _ = env.step(action)
    if done:
        break


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (3,) + inhomogeneous part.